# Connecting the trailer bills with the original budget bill
This file provides to connect the trailer bills with the original budget bill.

In [1]:
import pandas as pd

## 1. Combined three trailer bills

### Step 1: Load Datasets

In [2]:
files = [
    {"name": "AB102_2023_trailer_bill.csv", "label": "AB102"},
    {"name": "SB104_2023_trailer_bill.csv", "label": "SB104"},
    {"name": "SB105_2023_trailer_bill.csv", "label": "SB105"}
]

all_dfs = []

for f in files:
    try:
        # Read each CSV file into a pandas DataFrame
        df = pd.read_csv(f["name"])
        
        # Insert a new column at the beginning to track the bill source
        df.insert(0, "trailer_bill_source", f["label"])
        
        all_dfs.append(df)
        print(f"{f['label']} is read ({len(df)} rows)")
    except Exception as e:
        # Log error if file reading fails
        print(f"Error: Failed to read {f['label']}: {e}")

AB102 is read (767 rows)
SB104 is read (296 rows)
SB105 is read (21 rows)


### Step 2: Combine DataFrames

In [3]:
combined_df = pd.concat(all_dfs, ignore_index=True)

### Step 3: Filter by Budget Year

In [4]:
combined_df = combined_df[combined_df["budget_year"] != 2022]
combined_df = combined_df[combined_df["budget_year"] != "2022"]

### Step 4: Export Consolidated Data

In [5]:
print(f"\nConsolidated. Total rows: {len(combined_df)}")
# Save the final DataFrame without the index column
combined_df.to_csv("combined_trailer_bills.csv", index=False)
print("Combined file saved as 'combined_trailer_bills.csv'.")

# Preview the head of the consolidated DataFrame for verification
combined_df.head()


Consolidated. Total rows: 1079
Combined file saved as 'combined_trailer_bills.csv'.


,trailer_bill_source,level,budget_year,action_type,ref_source,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount
0,AB102,item,2023,Amended,Original,NaN,0250-001-0001,NaN,NaN,1,NaN,NaN,For support of Judicial Branch,620021000
1,AB102,program,2023,Amended,Original,NaN,0250-001-0001,130.0,NaN,1,NaN,1.0,Supreme Court,55790000
2,AB102,program,2023,Amended,Original,NaN,0250-001-0001,135.0,NaN,1,NaN,2.0,Courts of Appeal,271488000
3,AB102,program,2023,Amended,Original,NaN,0250-001-0001,140.0,NaN,1,NaN,3.0,Judicial Council,286639000
4,AB102,program,2023,Amended,Original,NaN,0250-001-0001,155.0,NaN,1,NaN,4.0,Habeas Corpus Resource Center,18597000


## 2. Connecting this csv file with the file of the original budget bill and dealing with the "added" and "repealed" items

### Step 1: Load Datasets

In [7]:
df_original = pd.read_csv("SB101_2023_cleaned.csv")
df_trailer = pd.read_csv("combined_trailer_bills.csv")

### Step 2: Key Standardization and Base Column Setup

In [8]:
# Define keys for merging to ensure "Repealed" or "Amended" items are correctly identified
keys = ["item_number", "program_code", "object_code", "schedule_group"]

# Standardize data types and handle empty values to prevent matching errors
for col in keys:
    df_original[col] = df_original[col].astype(str).str.strip().str.upper().replace(["NAN", "NONE", ""], "EMPTY")
    df_trailer[col] = df_trailer[col].astype(str).str.strip().str.upper().replace(["NAN", "NONE", ""], "EMPTY")

# Initialize base columns in the original dataframe
df_original["revised_amount"] = df_original["amount"]
df_original["trailer_bill_source"] = "Original"
df_original["action_type"] = "Original"

### Step 3: Process "Added" Items

In [9]:
# Filter for items newly added in the trailer bills
df_added = df_trailer[df_trailer["action_type"] == "Added"].copy()

if not df_added.empty:
    # Set the trailer bill amount as the revised amount
    df_added["revised_amount"] = df_added["amount"]
    
    # Set original amounts to 0 to indicate these items did not exist previously
    df_added["amount"] = 0
    if "amount_numeric" in df_added.columns:
        df_added["amount_numeric"] = 0
        
    # Align columns with the original dataframe for a clean concatenation
    cols_to_keep = df_original.columns.tolist()
    if "revised_amount" not in cols_to_keep:
        cols_to_keep.append("revised_amount")
    
    df_added = df_added.reindex(columns=cols_to_keep)
    
    # Append new items to create the master dataframe
    df_master = pd.concat([df_original, df_added], ignore_index=True)
else:
    df_master = df_original.copy()

### Step 4: Process "Repealed" Items

In [10]:
# Filter for items to be removed from the budget
df_rep = df_trailer[df_trailer["action_type"] == "Repealed"].copy()

if not df_rep.empty:
    # Identify unique item numbers designated for repeal
    repealed_items = df_rep["item_number"].unique().tolist()
    
    # Create a mask for all rows in the master data belonging to repealed items
    rep_mask = df_master["item_number"].isin(repealed_items)
    
    # Update these rows to reflect zero budget and mark as Repealed
    df_master.loc[rep_mask, "revised_amount"] = 0
    df_master.loc[rep_mask, "action_type"] = "Repealed"
    
    # Map the specific trailer bill source responsible for the repeal
    item_to_source = dict(zip(df_rep["item_number"], df_rep["trailer_bill_source"]))
    df_master.loc[rep_mask, "trailer_bill_source"] = df_master.loc[rep_mask, "item_number"].map(item_to_source)

    print(f"Items Repealed: {len(repealed_items)}")
    print(f"Total rows set to zero: {rep_mask.sum()}")

Items Repealed: 18
Total rows set to zero: 34


### Step 5: Export this Result

In [11]:
# Save the intermediate results (Added and Repealed processed) to CSV
df_master.to_csv("2023_SB101_step1_AR.csv", index=False, encoding='utf-8-sig')

print(f"Original rows: {len(df_original)}")
print(f"Final rows (after Added): {len(df_master)}")
print("\n--- Action Type Distribution ---")
print(df_master["action_type"].value_counts())

Original rows: 4482
Final rows (after Added): 4547

--- Action Type Distribution ---
action_type
Original    4448
Added         65
Repealed      34
Name: count, dtype: int64


## 3. Dealing with "amended" items

### Step 6: Process "Amended" Items (Full Item Replacement)

In [12]:
# Extract all items marked for amendment from the trailer bills
df_amended_src = df_trailer[df_trailer["action_type"] == "Amended"].copy()

if not df_amended_src.empty:
    amended_items = df_amended_src["item_number"].unique()
    
    # --- PHASE A: Neutralize Original Records in Master ---
    # Identify existing rows for amended items and set revised_amount to 0
    dropped_mask = df_master["item_number"].isin(amended_items)
    df_master.loc[dropped_mask, "revised_amount"] = 0
    df_master.loc[dropped_mask, "action_type"] = "Amended (Dropped/Old)"
    
    # Set history_order to 1 to ensure historical rows appear first during sorting
    df_master.loc[dropped_mask, "history_order"] = 1 
    
    # Map the latest trailer bill source to the superseded records
    item_to_source = df_amended_src.sort_values("trailer_bill_source").groupby("item_number")["trailer_bill_source"].last().to_dict()
    df_master.loc[dropped_mask, "trailer_bill_source"] = df_master.loc[dropped_mask, "item_number"].map(item_to_source)

    # --- PHASE B: Prepare Current Snapshot from Trailer Bill ---
    df_new_entries = df_amended_src.copy()
    # Rename 'amount' to 'revised_amount' to align with master schema
    df_new_entries = df_new_entries.rename(columns={"amount": "revised_amount"})
    # Original baseline is 0 for these new structure entries
    df_new_entries["amount_numeric"] = 0
    df_new_entries["action_type"] = "Amended (New/Current)"
    
    # Set history_order to 2 to ensure current snapshot follows superseded rows
    df_new_entries["history_order"] = 2 
    
    # --- PHASE C: Consolidation ---
    # Merge the updated structure back into the master dataframe
    df_master = pd.concat([df_master, df_new_entries], ignore_index=True)

# Default history_order to 0 for items that were not modified
df_master["history_order"] = df_master["history_order"].fillna(0)

### Step 7: Final Sorting and Clean-up

In [13]:
# Define the logical column order for the final audit-ready CSV
final_columns = [
    'level', 'agency', 'item_number', 'program_code', 'object_code', 
    'fund_code', 'fund_name', 'schedule_group', 'name', 
    'amount_numeric', 
    'revised_amount', 
    'trailer_bill_source', 
    'action_type'
]

# Primary Sort: Group by Item Number and Schedule, then sequence by History (Old -> New)
df_final = df_master[final_columns + ["history_order"]].copy()
df_final = df_final.sort_values(
    by=['item_number', 'schedule_group', 'history_order'], 
    ascending=[True, True, True] 
)

# Remove the internal 'history_order' column before final output
df_final = df_final[final_columns]

## 4. Save as a csv file

In [14]:
output_filename = "2023_budget_bill_final.csv"
df_final.to_csv(output_filename, index=False, encoding='utf-8-sig')

# Display execution summary for traceability
print(f"Original records: {len(df_original)}")
print(f"Final consolidated records: {len(df_final)}")
print("\n--- Final Action Type Distribution ---")
# Count of each action to verify how many items were Added, Repealed, or Amended
print(df_final["action_type"].value_counts())

print(f"\nProcessing complete. Final file saved as: {output_filename}")

Original records: 4482
Final consolidated records: 5543

--- Final Action Type Distribution ---
action_type
Original                 3665
Amended (New/Current)     996
Amended (Dropped/Old)     790
Added                      58
Repealed                   34
Name: count, dtype: int64

Processing complete. Final file saved as: 2023_budget_bill_final.csv
